# LLM Chains & Patterns
By using LLM chains, we can follow modular approach breaking down complex tasks into small managable steps. Each step can be implemented as independent and can be integrated in Chain based on functional requirement

Chains can be implemented using various patters

**1** Sequential Chains - like waterfall approach

**2** Router/Conditional Chain 

**3** Parallel Chains

**4** Map/Reduce Chains

**5** Conversational Chains

**6** Transformation Chains

**7** Batch Processing Chains




## LCEL -LangChain Expression Language
Recent version of LangChain moved to LCEL (LangChain Expression Language) which uses the pipe operator (|) for chaining. It's more flexible, composable, and provides better streaming support.

The LCEL approach is much more flexible and composable than the old LLMChain pattern.

In [13]:
# import packages
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


### Multiple Sequential Steps - Sequential chain pattern

In [12]:
llm = OllamaLLM(model="gemma3:1b", base_url="http://localhost:11434")

# Step 1 - Create a poem writer chain
poem_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "Write poem of 3-4 sentences about {topic}")
])
poemWriterChain = poem_prompt | llm | StrOutputParser() | RunnableLambda(lambda x: (print(f"Step 1 - Generated Poem: {x}\n"), x)[1])
                  
# Step 2 - Use the poem writer chain
teacher_promt = ChatPromptTemplate.from_messages([
    ("system", "You are a teacher who explains poems in simple terms."),
    ("user", "Explain {poem} in simple terms.")
])

teacherChain = teacher_promt | llm | StrOutputParser()

# Step 3 - Combine both chains
full_chain = {"poem": poemWriterChain} | teacherChain
response = full_chain.invoke({"topic": "Food"})

print(f"Final Explanation:\n{response}")

Step 1 - Generated Poem: Okay, here's a poem about food, three to four sentences long:

The flavors dance, a joyful hue,
From sweet berries to a savory stew,
Warm bread and fruit, a comforting view,
Food fills our hearts, forever true.

Final Explanation:
Okay, that’s a lovely poem! Let’s break it down.

It’s a poem about food – really, it’s about how delicious things are! The words “flavors dance” and “joyful hue” suggest that food is beautiful and exciting to look at. It talks about different kinds of food – berries, stew, bread, and fruit – and how they make us feel good. Basically, it’s a simple celebration of what we love to eat! 

Do you want to talk about *why* it’s a good poem, or maybe we could try another one?


### Router/Conditional Chain

In [25]:
from langchain_core.runnables import RunnableBranch

java_code_Prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Java programmer."),
    ("user", "Generate code for : {text}")
])

java_code_chain = java_code_Prompt | llm | StrOutputParser()


python_code_Prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful Python programmer."),
    ("user", "Generate code for : {text}")
])

python_code_chain = python_code_Prompt | llm | StrOutputParser()

branchChain = RunnableBranch(
    (lambda x: x["language"].lower() == "java", java_code_chain),    
    python_code_chain # Default case
)

result = branchChain.invoke({"text": "Hello World", "language": "Java"})

print(f"Branch Chain Result:\n{result}")

Branch Chain Result:
```java
public class HelloWorld {

  public static void main(String[] args) {
    System.out.println("Hello, World!");
  }
}
```

**Explanation:**

*   **`public class HelloWorld {`**:  This declares a public class named `HelloWorld`. In Java, all code must reside within a class.
*   **`public static void main(String[] args) {`**:  This is the main method.  It's the starting point of your Java program.  Here's a breakdown of the parts:
    *   `public`:  This makes the method accessible from anywhere.
    *   `static`: This means the method belongs to the class itself, not to any specific instance (object) of the class.
    *   `void`: This indicates that the method doesn't return any value.
    *   `main(String[] args)`:  The name of the method.  The `main` method is special; it's the entry point where the Java Virtual Machine (JVM) begins execution. The `String[] args` is an array of strings that allows you to pass command-line arguments to your program.
*   **`S

### Parallel Chains

In [30]:
from langchain_core.runnables import RunnableParallel

german_translate_Prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful German translator."),
    ("user", "Translate to German: {text}")
])
german_translate_chain = german_translate_Prompt | llm | StrOutputParser()

french_translate_Prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful French translator."),
    ("user", "Translate to French: {text}")
])
french_translate_chain = french_translate_Prompt | llm | StrOutputParser()

parallel_router = RunnableParallel({
    "german": german_translate_chain,
    "french": french_translate_chain
})


result = parallel_router.invoke({"text": "What is your name?"})

print(f"Parallel Chain Result:\n{result}")

Parallel Chain Result:
{'german': 'The most common and polite translation is:\n\n**Wie heißen Sie?**\n\n(pronounced: Veeh hießen sche?)\n\nIt’s a formal way of asking someone’s name. \n\nYou could also say:\n\n**Wie heißt du?** (pronounced: Veeh zay doo?) - This is more informal and used with people you know well.\n\nWhich one you use depends on the context and your relationship with the person.', 'french': 'Here are a few options for translating “What is your name?” into French, depending on the context and level of formality:\n\n**Most Common & Generally Appropriate:**\n\n*   **Comment vous appelez-vous ?** - This is the most standard and polite way to ask.\n\n**Other options (slightly more casual):**\n\n*   **Quel est ton nom ?** - This translates more literally to "What is your name?" and is perfectly acceptable.\n*   **Comment tu t\'appelles ?** - This is informal ("How do you call yourself?") and is appropriate for friends and family.\n\n**Therefore, my recommendation would be: C